# 04 — The final pipeline, routing and guardrails

The shipped assistant, and the acceptance tests that carry 30% of the grade.

## Order of operations, and why it is this order

1. **Guardrails first.** Out-of-scope carriers and live-data requests are caught
   *before* retrieval — deterministically, so a refusal never depends on the model
   choosing to behave, and cheaply, so we never embed a query we intend to refuse.
2. **Retrieve broadly, then route.** Retrieval runs unfiltered so the *evidence*
   decides the business line. Filtering first would require classifying the question
   up front, which is exactly what fails on cargo-flavoured financial questions.
3. **Scope the context to the routed line.** The anti-contamination step: once the
   route is decided, other business lines are dropped so the generator cannot quote
   the cargo tariff at a passenger.
4. **Abstain on weak evidence**, then generate with enforced citations.

In [ ]:
import sys, json; sys.path.insert(0, "../src")
import pandas as pd
from pathlib import Path
from delta_rag.pipeline import DeltaSupportBot

RESULTS = Path("../reports/results")
bot = DeltaSupportBot()
print("pipeline :", bot.config.label())
print("generator:", bot.model)
print("chunks   :", len(bot.chunks))

## Routing was measured, and the simplest policy won

In [ ]:
pd.DataFrame(json.loads((RESULTS / "stage0c_routing.json").read_text()))

I built two "better" routers and both were **worse** than trusting the top-1 retrieved
chunk. A rank-weighted vote over the top 5 scored 0.9143 and a vote plus a prior
extracted from the documents' own definitions sections scored 0.9429, against **1.0000
on the eval questions and 0.9714 overall for plain top-1**.

Both richer policies misroute R1 ("How much revenue did Delta make from shipping cargo
last year?") because lower-ranked cargo-tariff chunks outvote the single correct
financial record. The added machinery actively hurt, so it is off by default.

One known misroute remains: R2 ("liability limit for a damaged shipment") goes to the
passenger contract, whose baggage-liability clause is genuinely the most similar text
in the corpus and even quotes a dollar figure ($4,700/passenger). The defined-term
prior fixes R2 but breaks R1 — so shipping a fix validated on the single case it was
built for would be overfitting. It is documented as a limitation instead.

## The abstain threshold, and why it is deliberately loose

In [ ]:
cal = pd.DataFrame(json.loads((RESULTS / "guardrail_calibration.json").read_text()))
cal[(cal["Threshold"] >= 0.63) & (cal["Threshold"] <= 0.75)]

The gate runs on **dense cosine similarity**, not the fused RRF/weighted score. A
fusion score is a rank artefact with no absolute meaning and is not comparable across
queries, so no fixed threshold on it could ever be principled.

The two populations **overlap** — the hardest answerable question scores below the most
confidently-retrieved unanswerable probe — so no threshold separates them. It is set to
reject **no** answerable question rather than to maximise refusals, because this gate is
only the coarse outer layer. The precise layer is the system prompt, and probe G4 shows
it working on a question with *high* retrieval confidence:

> *"Delta's published documents do not cover fees for checked bags on transatlantic
> flights specifically. The rules provided are for domestic travel only [1]."*

Cosine similarity could never have caught that — the retrieved baggage rules genuinely
*are* the most topically similar text in the corpus. They just do not contain the
answer. Tightening the gate would buy refusals the prompt already handles, at the cost
of real answers.

## Acceptance tests

In [ ]:
acc = pd.DataFrame(json.loads((RESULTS / "acceptance_tests.json").read_text()))
print("required A1-A5:", (acc[acc['#'].str.startswith('A')]['Result'] == 'PASS').sum(), "/ 5")
print("all probes     :", (acc['Result'] == 'PASS').sum(), "/", len(acc))
acc[["#", "Question", "Expected", "Route", "Blocked", "Result", "Detail"]]

## Live demonstration

Four questions covering all three routes plus a guardrail. Note the third one — it is
lexically saturated with cargo vocabulary but must reach the financial filings.

In [ ]:
demo = [
    "What's Delta's policy on oversold flights?",
    "What are the packaging requirements for shipping cargo with Delta?",
    "How much revenue did Delta make from shipping cargo last year?",
    "What's JetBlue's baggage policy?",
]
for q in demo:
    r = bot.answer(q)
    print("=" * 90)
    print(f"Q: {q}")
    print(f"   route={r.route} blocked={r.blocked} reason={r.reason} "
          f"citations_ok={r.citations_valid} gen={r.generation_ms:.0f}ms")
    print(r.answer[:480])
    if r.sources:
        print("   sources:", [s["citation"] for s in r.sources][:3])

## Running the app

```bash
# terminal 1 — API (wait for "ready", ~20 s to embed the corpus and build FAISS)
.venv/bin/uvicorn backend.main:app --port 8000

# terminal 2 — UI
cd frontend && npm run dev      # http://localhost:5173
```

The UI shows the routed business line per answer, renders `[n]` citation markers as
chips linked to the source list, and reports retrieval/generation latency so the
pipeline's behaviour is visible rather than hidden.